# Line-by-line search

**Difficulty:** Beginner

Search a document by meaning, not keywords: tag each line with an ID, ask a `Choice` question to rank the lines by how well they answer a query, and a `Noul` question — in the same request — to check whether the document has an answer at all.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/cookbooks/semantic_find (click to expand)</summary>

# Line-by-line search

> Build semantic search for GitHub's Terms of Service. In one request, score 218 line ids against a plain-language query with a Choice question, and use a Noul question to check whether the document contains an answer.

You have a document and a plain-language question about it. You need the lines that answer the question, and a way to detect when the document has no answer.

The search backend comes together in three parts:

1. Tag each line with an ID so TypeSafe can point to it.
2. Use a `Choice` question to rank those line IDs by how well they answer the query. Choice probabilities always add up to 1, so a line ranks first even when none answer the query.
3. In the same request, use a `Noul` question to check whether the document contains an answer at all.

## Step 1: tag every line with an ID

The test document is GitHub's Terms of Service, split into 218 clauses. Each line gets a short ID prefix, and the lines are joined back into one document:

```
L052| You own Your Content. If you post Content you did not create, you are responsible for...
L053| You grant us and other Users the licenses in Sections D.4-D.8. These licenses apply...
L054| 4. License Grant to Us
```

## Step 2: ask where the answer is

A `Choice` question returns a probability for every option. Use the line IDs as the options, and "pick an option" becomes "point to a line":

```
where_question(query) = Choice(
    instructions=f'Which line of the document contains the answer to: "{query}"?',
    criteria={line_id(i): None for i in range(len(LINES))},
)
```

The option descriptions are `None` because the document already contains the text for each ID — the query goes in `instructions`, the state (the tagged document) stays unchanged between searches. (A `Choice` question accepts up to 255 options, so this recipe searches documents of up to 255 lines in one request; past that, search in two passes — one `Choice` picks a window of lines, a second ranks the lines inside it.)

## Step 3: check whether an answer exists

Choice probabilities always add up to 1, so *some* line ranks first even when the document doesn't answer the question at all. Ranking alone can't distinguish a real answer from the closest irrelevant line — so ask a second question, in the same request:

```
exists_question(query) = Noul(
    instructions=f'Does any line of the document address or answer: "{query}"?',
    criteria={
        "true": "At least one line of the document states or directly implies the answer",
        "false": "No line of the document addresses this",
    },
)
```

Unlike the Choice probabilities, the Noul probability doesn't depend on the other lines, so it can genuinely fall near zero when the document has no answer.

## Reading the result

`relevance` keeps one score per line (the Choice probabilities, in document order); `exists` is the Noul value. Local code applies thresholds to `exists` (the cookbook uses `>= 0.7` for "answered," `< 0.35` for "not in this document," in between for "partially addressed") — tune these against your own documents rather than trusting them as-is.

## What the scores mean, from the cookbook's own run against GitHub's ToS

- **"who owns the code I upload?"** — `exists 0.98`, top line scores `0.95`. A direct answer, confidently located.
- **"do I have to take disputes to arbitration?"** — the closest line scores `0.86` on relevance, but `exists` is only `0.14`. The ranking finds the *closest* line regardless; only the existence check reveals the document doesn't actually answer this.
- **"can minors use GitHub with parental permission?"** — the age rule ranks first, but it doesn't say whether parental permission changes anything. Partially addressed.

The ranking tells you *where* to look; the `exists` score tells you *whether* the result actually answers the question.

> The full cookbook fetches a real 218-line, ~44,000-character GitHub Terms of Service document from a pinned gist — trimmed here to a handful of lines, since the mechanism doesn't need the full document. Read the full cookbook at [docs.typesafe.ai/cookbooks/semantic_find](https://docs.typesafe.ai/cookbooks/semantic_find) any time.

</details>

## The core idea

Two primitives, one request, two different jobs: `Choice` ranks (which line is closest?), `Noul` verifies (does any line actually answer this at all?). Neither one alone is enough — `Choice`'s probabilities always sum to 1, so it always has *an* opinion about which line is best, even when every line is irrelevant. `Noul` is the independent check that can genuinely say "none of these," because its probability doesn't have to add up to anything.

This is the same "ask everything in one request" shape as speculative fan-out, just with the two questions playing complementary roles instead of covering different branches of a decision tree.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { choice, noul } from "npm:@typesafe-ai/sdk";

const lines = [
  "You own any code you upload, but you grant us a license to host, display, and share it as needed to provide the service.",
  "We may suspend or terminate your account at any time, for any reason, without prior notice.",
  "You must be at least 13 years old to use this service.",
  "Refunds are issued within 5 business days of a cancellation request.",
  "This service is provided as-is, with no warranty of any kind, express or implied.",
];

function lineId(i: number): string {
  return `L${String(i).padStart(2, "0")}`;
}

const document = lines.map((line, i) => `${lineId(i)}| ${line}`).join("\n");

async function find(query: string) {
  const response = await client.systemOne({
    state: document,
    questions: {
      where: choice(
        `Which line of the document contains the answer to: "${query}"?`,
        Object.fromEntries(lines.map((_, i) => [lineId(i), null])),
      ),
      exists: noul(`Does any line of the document address or answer: "${query}"?`, {
        true: "At least one line of the document states or directly implies the answer",
        false: "No line of the document addresses this",
      }),
    },
  });

  const { where, exists } = response.answers;
  return {
    topLine: where.choice,
    topProbability: where.probabilities[where.choice],
    exists: exists.noul,
  };
}

console.log(await find("can I get my money back?"));
console.log(await find("what happens if I violate the rules?"));

## What just happened

Two queries against the same five-line mini-document. "Can I get my money back?" should point at the refund line with a high `exists` score — a direct hit. "What happens if I violate the rules?" is vaguer: there's a termination clause, but nothing that specifically says "if you break the rules" — expect a lower `exists` than the refund query, even though `Choice` will still confidently point at *some* line.

**Try it yourself:** ask a query this document genuinely can't answer at all (e.g. "what payment methods do you accept?") and predict how low `exists` drops, and which line `Choice` picks anyway — remember, it always has to pick something.

## Related

- `lessons/core-concepts/0002-choice.ipynb`, `lessons/core-concepts/0004-noul-and-picking-the-right-primitive.ipynb` — the two primitives this technique combines
- `cookbooks/how-to/re-ranking.ipynb` — a related way to search a corpus, scoring each candidate independently instead of ranking them against each other in one `Choice`

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/typesafe-ai.md`.